## Monter Google Drive (Colab)

In [5]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Vérifier TensorFlow

In [6]:
import tensorflow as tf

print("="*50)
print("Version TensorFlow :", tf.__version__)
print("="*50)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU détecté :", gpus)
else:
    print("Aucun GPU détecté")

Version TensorFlow : 2.20.0
GPU détecté : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## import des biblio

In [7]:
import os
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model

from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

## definition des chemin dans drive

In [8]:
BASE_PATH = "/content/drive/MyDrive/Detection_Deforestation_IA"

DATA_PATH = os.path.join(
    BASE_PATH,
    "data"
)

MODEL_PATH = os.path.join(
    BASE_PATH,
    "models"
)


TRAIN_PATH = os.path.join(
    DATA_PATH,
    "train"
)

VALIDATION_PATH = os.path.join(
    DATA_PATH,
    "validation"
)

TEST_PATH = os.path.join(
    DATA_PATH,
    "test"
)


print("Train :", TRAIN_PATH)
print("Validation :", VALIDATION_PATH)
print("Test :", TEST_PATH)
print("Models :", MODEL_PATH)

Train : /content/drive/MyDrive/Detection_Deforestation_IA/data/train
Validation : /content/drive/MyDrive/Detection_Deforestation_IA/data/validation
Test : /content/drive/MyDrive/Detection_Deforestation_IA/data/test
Models : /content/drive/MyDrive/Detection_Deforestation_IA/models


## Vérification des dossiers

In [9]:
print(os.listdir(DATA_PATH))

print("\nTrain :")
print(os.listdir(TRAIN_PATH))

print("\nValidation :")
print(os.listdir(VALIDATION_PATH))

print("\nTest :")
print(os.listdir(TEST_PATH))

['test', 'train', 'validation']

Train :
['forest', 'non_forest']

Validation :
['non_forest', 'forest']

Test :
['forest', 'non_forest']


## Charger le modèle CNN construit dans Notebook 03

In [10]:
model = load_model(
    os.path.join(
        MODEL_PATH,
        "cnn_initial.keras"
    )
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,914,309 (37.82 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,609,540 (25.21 MB)

## Création des générateurs d'images

In [11]:
train_datagen = ImageDataGenerator(
    rescale=1./255
)


validation_datagen = ImageDataGenerator(
    rescale=1./255
)


test_datagen = ImageDataGenerator(
    rescale=1./255
)

## Chargement des images

In [12]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode="binary"
)


validation_generator = validation_datagen.flow_from_directory(
    VALIDATION_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode="binary"
)


test_generator = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

Found 5600 images belonging to 2 classes.
Found 1200 images belonging to 2 classes.
Found 1200 images belonging to 2 classes.


## Vérifier les classes

In [13]:
print(train_generator.class_indices)

{'forest': 0, 'non_forest': 1}


## Création des callbacks

In [14]:
checkpoint = ModelCheckpoint(
    filepath=os.path.join(
        MODEL_PATH,
        "best_model.keras"
    ),
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)


early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)


reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

## Entraînement du CNN

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)

Epoch 1/20
170/175 ━━━━━━━━━━━━━━━━━━━━ 34s 7s/step - accuracy: 0.7708 - loss: 0.4480

## Courbe Accuracy

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history["accuracy"],
    label="Train"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation"
)

plt.title("Accuracy du modèle")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend()
plt.grid()

plt.show()

## Courbe Loss

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history["loss"],
    label="Train"
)

plt.plot(
    history.history["val_loss"],
    label="Validation"
)

plt.title("Loss du modèle")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.grid()

plt.show()

## Évaluation sur test

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_generator
)

print("Test Loss :", test_loss)
print("Test Accuracy :", test_accuracy)

## Rapport classification

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)


test_generator.reset()


predictions = model.predict(
    test_generator
)


y_pred = (
    predictions > 0.5
).astype(int)


y_true = test_generator.classes


print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "forest",
            "non_forest"
        ]
    )
)

## Matrice de confusion

In [ ]:
import seaborn as sns


cm = confusion_matrix(
    y_true,
    y_pred
)


plt.figure(figsize=(6,5))


sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=[
        "forest",
        "non_forest"
    ],
    yticklabels=[
        "forest",
        "non_forest"
    ]
)


plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion")


plt.show()

## sauvegarde final du model

In [ ]:
model.save(
    os.path.join(
        MODEL_PATH,
        "deforestation_cnn_final.keras"
    )
)


print(" Modèle final sauvegardé dans Google Drive")